# 02b · LIBERO eval — seed 2 (150k, 100ep, 외란 X)
seed2 의 **150k 체크포인트**를 순수 LIBERO eval (4모델을 GPU 0-3 에 하나씩).
- 에피소드 = **100**, 체크포인트 = **150k**, 외란 없음(`lerobot_eval` 직접, action(.pt)=jerk 기록됨).
- seed별 분리 → 다른 머신에서 seed1/2/3 동시 실행 가능. 결과 = `eval_clean_dir` → `07_report_sr` 자동 pooled.
- ⚠️ LIBERO 시뮬 필요(`00b`). eval GPU 는 비어 있어야 함(학습과 겹치면 OOM).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

SEED      = 2
N_EP      = 100
CKPT_STEP = 150000   # 150k
print('seed:', SEED, '| n_episodes:', N_EP, '| ckpt:', CKPT_STEP)
print('models:', cf.FINAL_TAGS, '| task:', cf.PRIMARY_SIM,
      '| LIBERO_EVAL_BATCH:', cf.LIBERO_EVAL_BATCH, '(동시 env =', cf.LIBERO_EVAL_BATCH * 10, ')')

## 1) 사전 체크 — 150k 체크포인트가 있나
`⚠`/`❌` 뜨면 그 모델은 아직 150k 미도달.

In [ ]:
ok = True
for t in cf.FINAL_TAGS:
    got = cf.resolved_ckpt_step(t, SEED, step=CKPT_STEP)
    if got == CKPT_STEP:
        flag = 'OK'
    elif got is None:
        flag = '\u274c ckpt 없음'; ok = False
    else:
        flag = f'\u26a0 150k 없음 \u2192 최근접 {got:,} 사용'; ok = False
    print(f'  {t:12} seed{SEED}: {flag}')
print('\n\u2705 4모델 전부 150k 존재' if ok else '\n\u26a0 일부 150k 아님 — 위 확인 후 진행')

## 2) 실행 — 4모델을 GPU 0-3 에 하나씩 (동시)
`launch_cmds_live` 가 4개 끝날 때까지 대기. GPU 당 eval 1개 → 겹침/OOM 없음.

In [ ]:
labeled = []
for i, t in enumerate(cf.FINAL_TAGS):          # 4모델 -> GPU 0..3
    try:
        cmd = cf.libero_eval_cmd(t, seed=SEED, gpu_id=i, n_episodes=N_EP, step=CKPT_STEP)
        labeled.append((f'{t}/seed{SEED}/{N_EP}ep', cmd))
    except FileNotFoundError as e:
        print('  skip:', e)
print(f'===== seed{SEED}: {N_EP}ep, ckpt {CKPT_STEP:,} — {len(labeled)}잡 (GPU 0-3) =====')
if labeled:
    cf.launch_cmds_live(labeled, log_tag=f'eval150k_seed{SEED}')
print('\n\U0001f3c1 seed', SEED, 'eval 완료')

## 3) 결과 (SR + 저장된 영상 개수)
영상은 standalone `lerobot_eval` 이 `eval_clean_dir/**/*.mp4` 로 저장(fork 기본 3개).

In [ ]:
import glob
print(f"{'MODEL':<14}{'SEED':>5}{'EP':>6}{'SR':>8}{'VIDEOS':>8}")
print('-' * 42)
for t in cf.FINAL_TAGS:
    st = cf.get_eval_status(t, SEED, cf.PRIMARY_SIM)
    sr = f"{st['sr']*100:.1f}%" if st['sr'] is not None else '-'
    out = cf.eval_clean_dir(t, SEED, cf.PRIMARY_SIM)
    vids = glob.glob(str(out / '**' / '*.mp4'), recursive=True)
    print(f'{t:<14}{SEED:>5}{N_EP:>6}{sr:>8}{len(vids):>8}')
# 영상 경로 예시
_ex = cf.eval_clean_dir(cf.FINAL_TAGS[0], SEED, cf.PRIMARY_SIM)
print(f'\n영상 위치: {_ex}/videos/  (모델별)')
print('※ 영상 0개면 = fork 가 standalone eval 영상도 껐다는 뜻 → 알려주면 fork max_episodes_rendered 패치 안내')